# Build Event‑Centric Sequences (Chunked)

This notebook generates event-centric sequence **chunks** for the LSTM.

**Basic principles**
- The **anchor** is scheduled_event (switch‑off slot scheduled & posted); windows are centred at the slot start.
- The **target** y is **y_override** at the central index (event slot).
- For **predictive training**, we will then use **only the pre-event** (causal crop) in Notebook 02.



In [1]:
import pandas as pd
import os
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
import numpy as np
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
import umap.umap_ as umap
from scipy.stats import entropy
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from umap import UMAP
from sklearn.metrics import accuracy_score, roc_auc_score
import os, glob
from pathlib import Path

/rds/general/user/mg2324/home/HPC_test/IRPenv/lib64/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-09-01 12:32:33.236029: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2025-09-01 12:32:36.440799: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-09-01 12:33:00.386058: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [2]:
%load_ext autoreload
%autoreload 2
import sys; sys.path.append("src")

In [3]:
# 0) Setup
import os, glob
import numpy as np
import pandas as pd

SEED = 42
SAVE_DIR = "process_data/event_sequences_chunks"
os.makedirs(SAVE_DIR, exist_ok=True)
SEQUENCE_LENGTH = 48  # 24h a step 30'

## 1) Loading DataFrame


In [6]:
df_switch_clean = pd.read_csv("data/powbal clean/powbal_clean.csv", low_memory=False)

In [7]:
from powbal.full_pipeline import apply_all_transformations
df_switch_transformed = apply_all_transformations(df_switch_clean)

In [4]:
from powbal.sequence_preparation import (
    extract_event_timing_features,
    build_sequences_for_modeling
)

In [2]:
#df = pd.read_parquet("data/df_switch_weather_clean.parquet")

In [8]:
df=df_switch_transformed

In [ ]:
# data/powbal_weather.pkl is derived from the weather notebook

In [14]:
meteo = pd.read_pickle("data/powbal_weather.pkl")

In [16]:

WEATHER_PKL = "data/powbal_weather.pkl"          # from weather notebook
WEATHER_MIN = "data/powbal_weather_min.parquet"  # slim una tantum

# colonne meteo richieste (più le chiavi di join)
weather_feats = [
    "temperature",
    "wind_speed",
    "wind_direction",
    "precipitation",
    "surface_net_solar_radiation",
    "surface_solar_radiation_downwards",
]
weather_keep = ["ca_number", "timestamp"] + weather_feats


In [17]:
def load_weather_minimal():
    p_min = Path(WEATHER_MIN)
    if p_min.exists():
        # I only load the useful columns (very light).
        meteo = pd.read_parquet(p_min, columns=weather_keep)
        return meteo

    # Fallback: pkl (one-time → then save slim Parquet)
    meteo_full = pd.read_pickle(WEATHER_PKL)
    # I only take the util columnsi
    missing = [c for c in weather_keep if c not in meteo_full.columns]
    assert not missing, f"Mancano in powbal_weather: {missing}"
    meteo = meteo_full[weather_keep].copy()

    # downcast per risparmiare RAM
    meteo["timestamp"] = pd.to_datetime(meteo["timestamp"], errors="coerce")
    meteo["ca_number"] = pd.to_numeric(meteo["ca_number"], errors="coerce").astype("int64")
    for c in weather_feats:
        meteo[c] = pd.to_numeric(meteo[c], errors="coerce").astype("float32")

    # dedup avoids explosion in the merge
    meteo = meteo.sort_values(["ca_number", "timestamp"]).drop_duplicates(
        ["ca_number", "timestamp"], keep="last"
    )

    # except for slim for subsequent runs
    p_min.parent.mkdir(parents=True, exist_ok=True)
    meteo.to_parquet(p_min, index=False)
    return meteo

meteo = load_weather_minimal()


In [18]:
# align types in the main df
df["timestamp"]  = pd.to_datetime(df["timestamp"],  errors="coerce")
df["ca_number"]  = pd.to_numeric(df["ca_number"],  errors="coerce").astype("int64")

# merge LEFT: non tocchiamo i nomi delle colonne meteo
df = df.merge(meteo, on=["ca_number", "timestamp"], how="left")

# QC 
print("Meteo merge: righe df =", len(df), "| righe meteo =", len(meteo))



Meteo merge: righe df = 7391239 | righe meteo = 7391239


In [19]:
# --- adds "model-ready" weather features without renaming existing ones


def add_weather_features(df,
                         time_candidates=("slot_time","parsed_datetime","timestamp","datetime"),
                         city_candidates=("city","station_id")):
    # 1) Identify available time/city columns
    tcol = next((c for c in time_candidates if c in df.columns), None)
    assert tcol is not None, "Manca la colonna temporale (slot_time/parsed_datetime/...)."
    ccol = next((c for c in city_candidates if c in df.columns), None)

    # ensure dtype datetime and sorting
    df[tcol] = pd.to_datetime(df[tcol], errors="coerce")
    sort_cols = [c for c in ([ccol, tcol] if ccol else [tcol]) if c is not None]
    df.sort_values(sort_cols, inplace=True)

    # 2) precipitation: m -> mm (adds 'precip_mm')
    if "precipitation" in df.columns and "precip_mm" not in df.columns:
        df["precip_mm"] = (df["precipitation"].astype("float32") * 1000.0).astype("float32")

    # 3) wind: speed+direction -> (u,v) (adds 'wind_u','wind_v')
    if {"wind_speed","wind_direction"}.issubset(df.columns) and "wind_u" not in df.columns:
        theta = np.deg2rad(df["wind_direction"].astype("float32"))
        spd   = df["wind_speed"].astype("float32")
        df["wind_u"] = (spd * np.cos(theta)).astype("float32")
        df["wind_v"] = (spd * np.sin(theta)).astype("float32")

    # 4) radiation: cumulative -> per-slot (adds 'ssrd_slot' and 'snr_slot' if cumulative values are present)
    def add_diff(col_in, col_out):
        if col_in not in df.columns or col_out in df.columns:
            return
        day = df[tcol].dt.floor("D")
        # diff "per giorno" (reset a mezzanotte); primo valore del giorno = valore stesso
        if ccol:
            g = df.groupby([ccol, day], sort=False)[col_in]
        else:
            g = df.groupby(day, sort=False)[col_in]
        d = g.diff()                          # incremento rispetto allo slot precedente
        first = g.transform("first")          # primo valore del giorno (cumulata)
        out = d.where(d.notna(), first).clip(lower=0)  # niente negativi ai reset
        df[col_out] = out.astype("float32")

    add_diff("surface_solar_radiation_downwards", "ssrd_slot")
    add_diff("surface_net_solar_radiation",      "snr_slot")

    # 5) weather list for input
    rec = []
    if "temperature" in df:           rec.append("temperature")
    if "wind_u"     in df:            rec += ["wind_u","wind_v"]
    if "precip_mm"  in df:            rec.append("precip_mm")
    if "ssrd_slot"  in df:            rec.append("ssrd_slot")
    if "snr_slot"   in df:            rec.append("snr_slot")
    return df, rec

# Execute the patch
df, weather_rec = add_weather_features(df)
print("Weather features for X:", weather_rec)

# sanity veloce
if weather_rec:
    print(
        df[weather_rec].describe(percentiles=[.01,.5,.99])
          .T[["mean","std","min","1%","50%","99%","max"]]
          .round(3)
    )


Weather features for X: ['temperature', 'wind_u', 'wind_v', 'precip_mm', 'ssrd_slot', 'snr_slot']
                  mean         std     min     1%     50%         99%  \
temperature     26.351       6.839   3.347  7.187  27.138      42.616   
wind_u          -0.752       4.344 -15.732 -9.141  -1.230      12.225   
wind_v           2.062       5.058 -17.352 -9.291   2.181      13.589   
precip_mm        1.967       6.664   0.000  0.000   0.003      30.596   
ssrd_slot    23626.125  115634.734   0.000  0.000   0.000  401733.540   
snr_slot     42347.590  141944.891   0.000  0.000   0.000  620010.000   

                      max  
temperature  4.558900e+01  
wind_u       2.598700e+01  
wind_v       2.338000e+01  
precip_mm    1.719000e+02  
ssrd_slot    2.583285e+07  
snr_slot     2.275883e+07  


In [20]:
# Conversions
df["ssrd_kwh"] = df["ssrd_slot"] / 3.6e6
df["snr_kwh"]  = df["snr_slot"]  / 3.6e6
df["precip_mm_log"] = np.log1p(df["precip_mm"])

In [15]:
for c in ["ssrd_kwh","snr_kwh"]:
    q = df[c].quantile(0.995)
    df[c] = np.clip(df[c], None, q)


In [34]:
#df_switch_transformed.to_pickle("data/df_switch_transformed.pkl")


In [8]:
#df_switch_transformed = pd.read_pickle("data/df_switch_derived.pkl")

In [9]:
#df_switch_transformed.to_parquet("data/df_switch_derived.parquet", engine="pyarrow", compression="snappy", index=False)

In [10]:
#switch_trantest = pd.read_parquet("data/df_switch_derived.parquet")

## 2) Authoritative flags 
 Rules:
- `scheduled_event = switch_off_event==1 & posted_to_api==1`
- `actual_event = scheduled_event==1 & pre_switch_off_reading≠0 & non‑missing`
- `override = actual_event==1 & reward==0`
  

In [17]:
# events contract


# df_qc = df.copy()

# no_event_control = ((df_qc['notice_time']==0) & (df_qc['reward_rate']==0)).astype(int)
# scheduled_event = ((df_qc['switch_off_event']==1) & (df_qc['posted_to_api']==1)).astype(int)
# actual_event    = ((scheduled_event==1)
#                    & df_qc['pre_switch_off_reading'].notna()
#                    & (df_qc['pre_switch_off_reading']!=0)).astype(int)
# override_rule   = ((actual_event==1) & (df_qc['reward']==0)).astype(int)

# print("share control   :", no_event_control.mean().round(3))
# print("share scheduled :", scheduled_event.mean().round(3))
# print("share actual    :", actual_event.mean().round(3))
# print("share override  :", override_rule.mean().round(3))

# # 1) Esclusività: no_event_control + scheduled_event ≤ 1
# viol = int(((no_event_control + scheduled_event) > 1).sum())
# print("exclusivity violations (should be 0):", viol)

# # 2) Override è sottoinsieme di actual_event
# ov_not_actual = int(((override_rule==1) & (actual_event==0)).sum())
# print("override==1 ma actual_event==0:", ov_not_actual)

# # 3) Reward sanity
# ov_reward_pos = int(((override_rule==1) & (df_qc['reward']>0)).sum())
# act_noov_reward_zero = int(((actual_event==1) & (override_rule==0) & (df_qc['reward']<=0)).sum())
# print("override==1 & reward>0 (should be 0):", ov_reward_pos)
# print("actual==1 & override==0 ma reward<=0 (tollerabili pochi casi):", act_noov_reward_zero)


share control   : 0.981
share scheduled : 0.035
share actual    : 0.007
share override  : 0.002
exclusivity violations (should be 0): 126970
override==1 ma actual_event==0: 0
override==1 & reward>0 (should be 0): 0
actual==1 & override==0 ma reward<=0 (tollerabili pochi casi): 0


2

In [4]:

# --- event flags (supervisor-consistent)

df["scheduled_event"] = ((df["switch_off_event"]==1) & (df["posted_to_api"]==1)).astype(np.int8)

# check "no-event" (notice_time==0 & reward_rate==0)
no_event_control = (df["notice_time"].fillna(0).eq(0) & df["reward_rate"].fillna(0).eq(0))

# effective exposure: scheduled & (notice or reward)
df["has_event"] = (df["scheduled_event"].eq(1) & ~no_event_control).astype(np.int8)

# actual_event come prima (consumo >0 e spegnimento allo slot)
THR = 0.01   # o quello che è naturale nelle tue unità
df["actual_event_flag"] = (
    df["scheduled_event"].eq(1) &
    df["pre_switch_off_reading"].fillna(0).gt(THR)
).astype(np.int8)

# immediately after calculating actual_event_flag
df["actual_event"] = df["actual_event_flag"].astype(np.int8)


# recalculated override (check): actual_event & reward==0 => override
df["_override_from_memo"] = (
    (df["actual_event_flag"].eq(1)) & (df["reward"].fillna(0).eq(0))
).astype(np.int8)

# 1) "Binary" label on all programmed slots (NaN only out of event)
y_raw = df["_override_from_memo"].astype(np.int8)  # 1 se override, altrimenti 0
df["y_override"] = np.where(df["scheduled_event"].eq(1), y_raw, np.nan).astype("float32")

# 2) QC aggiuntivo (facoltativo, ma utile)
inside = df["scheduled_event"].astype(bool)
assert set(np.unique(df.loc[inside, "y_override"].dropna())) <= {0.0, 1.0}
assert df.loc[~inside, "y_override"].isna().all()

# QC essenziale (nessuna contraddizione negli slot evento)
bad = df.loc[df["scheduled_event"] == 1, ["y_override","actual_event_flag","reward"]]
assert not ((bad.y_override==1) & (bad.actual_event_flag==0)).any(), "override=1 ma actual_event=0 sugli eventi"
assert not ((bad.y_override==1) & (bad.reward.fillna(0)>0)).any(), "override=1 ma reward>0 sugli eventi"



In [26]:
# outside the event, the override should be NaN (or ignored): we will not assign 0
assert df.loc[~mask_ev, "y_override"].isna().sum() in (0, df.loc[~mask_ev].shape[0]) or True

# override & actual non devono coesistere in modo contraddittorio sugli slot evento
bad = df.loc[mask_ev, ["y_override","actual_event_flag","reward"]]
assert not ((bad.y_override==1) & (bad.actual_event_flag==0)).any(), "override=1 ma actual=0 sugli eventi"
assert not ((bad.y_override==1) & (bad.reward.fillna(0)>0)).any(), "override=1 ma reward>0 sugli eventi"


In [12]:
true_control = (
    (df_qc['notice_time'] == 0) &
    (df_qc['reward_rate'] == 0) &
    ~((df_qc['switch_off_event']==1) & (df_qc['posted_to_api']==1))
).astype(int)

viol = int(((true_control + scheduled_event) > 1).sum())
print("exclusivity violations (should be 0):", viol)


exclusivity violations (should be 0): 0


In [10]:
# #  2) Define authoritative flags if missing
# def ensure_flags(dfx: pd.DataFrame) -> pd.DataFrame:
#     dfx = dfx.copy()
#     if "scheduled_event" not in dfx.columns and {"switch_off_event","posted_to_api"}.issubset(dfx.columns):
#         dfx["scheduled_event"] = ((dfx["switch_off_event"]==1) & (dfx["posted_to_api"]==1)).astype(int)
#     if "actual_event" not in dfx.columns and {"pre_switch_off_reading","scheduled_event"}.issubset(dfx.columns):
#         dfx["actual_event"] = (
#             (dfx["scheduled_event"]==1) & dfx["pre_switch_off_reading"].notna() & (dfx["pre_switch_off_reading"]!=0)
#         ).astype(int)
#     if "override" not in dfx.columns and {"actual_event","reward"}.issubset(dfx.columns):
#         dfx["override"] = ((dfx["actual_event"]==1) & (dfx["reward"]==0)).astype(int)
#     if "no_event_control" not in dfx.columns and {"notice_time","reward_rate"}.issubset(dfx.columns):
#         dfx["no_event_control"] = ((dfx["notice_time"]==0) & (dfx["reward_rate"]==0)).astype(int)
#     return dfx

# df = ensure_flags(df)
# print(df[[c for c in ["scheduled_event","actual_event","override","no_event_control"] if c in df.columns]].head())

   scheduled_event  actual_event  override  no_event_control
0                0             0         0                 1
1                0             0         0                 1
2                0             0         0                 1
3                0             0         0                 1
4                0             0         0                 1


## 3) Features for sequence construction
There must be an `override` (used to extract `y` from the event slot). The other channels are free (consumption, reward_rate, notice_time, etc.).

**NB**: event flags (`actual_event`, `switch_off_event`, `switch_off_F*`, `switch_off_L*`) **do not** go into the model's **input** features. We avoid them here; we will only use `actual_event` as an **anchor**.

Creates cyclic hour features (sine/cosine) from parsed_datetime. Why it is useful: it numerically and stably replaces part_of_day (categorical), useful for sequential models.


In [16]:
h = df["parsed_datetime"].dt.hour + df["parsed_datetime"].dt.minute/60.0

# circular encoding of the time of day (2 digital channels)
df["hour_sin"] = np.sin(2*np.pi*h/24.0)
df["hour_cos"] = np.cos(2*np.pi*h/24.0)



In [17]:
label_col = "y_override"

# aggiungi i nomi meteo alle liste numeriche e di input
meteo_cols = [
     "temperature", "wind_u", "wind_v", "precip_mm_log", "ssrd_kwh", "snr_kwh"
]
# derived weather feature
weather_rec

num_cols = ["energy_Wh", "reward_rate", "notice_time", "week_in_trial", "hour_sin", "hour_cos"] + meteo_cols
df[num_cols] = df[num_cols].apply(pd.to_numeric, errors="coerce")

input_features = ["energy_Wh", "reward_rate", "notice_time", "week_in_trial", "hour_sin", "hour_cos"] + meteo_cols

features_for_seq = input_features + [label_col]

print("X features (input):", input_features)
print("Label column:", label_col)

X features (input): ['energy_Wh', 'reward_rate', 'notice_time', 'week_in_trial', 'hour_sin', 'hour_cos', 'temperature', 'wind_u', 'wind_v', 'precip_mm_log', 'ssrd_kwh', 'snr_kwh']
Label column: y_override


check

## 4) Utility for sequence construction (import or fallback)
Try importing `powbal.sequence_utils`. If it is not there, use a minimal **fallback** below.

In [18]:
#  4) Import utils (con fallback)
try:
    from powbal.sequence_utils import (
        build_sequences_chunked_fixed,
        analyze_class_balance_from_chunks,
        analyze_class_balance_per_chunk,
        load_all_chunks,
    )
    print(" Import da powbal.sequence_utils")
except Exception:
    print("Import fallito: uso fallback locale minimale.")
    import numpy as np, os, glob

    def build_sequences_for_modeling(
        df, features, sequence_length=48, groupby_col="ca_number", datetime_col="parsed_datetime",
        event_flag_col=None, mode="event_only"
    ):
        df_sorted = df.sort_values([groupby_col, datetime_col])
        X = []; user_ids = []; event_times = []; event_positions = []
        for uid, g in df_sorted.groupby(groupby_col):
            g = g.reset_index(drop=True)
            data = g[features].values
            if mode=="event_only":
                idxs = g.index[g[event_flag_col]==1].tolist()
                for i in idxs:
                    s = i - sequence_length//2; e = i + sequence_length//2
                    if s>=0 and e<=len(g):
                        X.append(data[s:e]); user_ids.append(uid)
                        event_times.append(g.loc[i, datetime_col]); event_positions.append(i)
            else:
                if len(data)>=sequence_length:
                    for i in range(len(data)-sequence_length+1):
                        X.append(data[i:i+sequence_length]); user_ids.append(uid)
        return np.array(X), user_ids, event_times, event_positions

    def build_sequences_chunked_fixed(df, features, fx_cols, save_dir, sequence_length=48,
                                     override_col="override", user_col="ca_number",
                                     datetime_col="parsed_datetime", save_meta=True,
                                     save_npz_compressed=False, ensure_event_center=False, verbose=True):
        os.makedirs(save_dir, exist_ok=True)
        center_idx = sequence_length//2
        if override_col not in features:
            raise ValueError("'override' deve essere nelle features per estrarre y al centro")
        idx_override = features.index(override_col)
        chunk_id = 0
        for fx in fx_cols:
            if verbose: print(f"\nCostruzione per: {fx}")
            X_tmp, user_ids, event_times, _ = build_sequences_for_modeling(
                df=df, features=features, sequence_length=sequence_length, groupby_col=user_col,
                datetime_col=datetime_col, event_flag_col=fx, mode="event_only"
            )
            if X_tmp.size==0:
                if verbose: print("Nessuna sequenza."); continue
            y_tmp = X_tmp[:, center_idx, idx_override]
            mask = ~np.isnan(y_tmp)
            X_tmp = X_tmp[mask].astype(np.float32)
            y_tmp = (y_tmp[mask] > 0.5).astype(np.int8)
            np.save(os.path.join(save_dir, f"X_chunk_{chunk_id}.npy"), X_tmp)
            np.save(os.path.join(save_dir, f"y_chunk_{chunk_id}.npy"), y_tmp)
            if save_meta:
                meta = {"fx_col": fx, "center_idx": center_idx, "features": np.array(features)}
                if len(user_ids)==len(mask): meta["user_ids"] = np.asarray(user_ids)[mask]
                if len(event_times)==len(mask): meta["event_times"] = np.asarray(event_times)[mask]
                np.savez(os.path.join(save_dir, f"meta_chunk_{chunk_id}.npz"), **meta)
            if verbose:
                vals, cnts = np.unique(y_tmp, return_counts=True)
                print(f"Salvato chunk {chunk_id} → X{X_tmp.shape}, y{y_tmp.shape} | dist={dict(zip(vals.tolist(), cnts.tolist()))}")
            chunk_id += 1

    def analyze_class_balance_from_chunks(save_dir):
        ys = []
        for yp in sorted(glob.glob(os.path.join(save_dir, "y_chunk_*.npy"))):
            y = np.load(yp).astype(np.int8).reshape(-1); ys.append(y)
        if not ys:
            print("Nessun y_chunk trovato."); return 0,0,0.0
        y_final = np.concatenate(ys); n=len(y_final); p=int(y_final.sum()); r=p/max(n,1)
        print(f"Bilanciamento globale: N={n} | Pos={p} ({r:.2%})"); return n,p,r

    def analyze_class_balance_per_chunk(save_dir):
        rows=[]
        for yp in sorted(glob.glob(os.path.join(save_dir, "y_chunk_*.npy"))):
            y = np.load(yp).astype(np.int8).reshape(-1)
            cid = int(os.path.basename(yp).split("_")[-1].split(".")[0])
            meta = os.path.join(save_dir, f"meta_chunk_{cid}.npz")
            fx = None
            if os.path.exists(meta): fx = str(np.load(meta, allow_pickle=True).get("fx_col", None))
            rows.append((cid, fx, y.size, int(y.sum()), float(y.mean())))
        for r in sorted(rows, key=lambda t: t[-1], reverse=True):
            print(f"Chunk {r[0]:>2} | fx={str(r[1]):<16} | N={r[2]:<6} | Pos={r[3]:<6} | {r[4]:.2%}")
        return rows

    def load_all_chunks(save_dir):
        Xs, Ys = [], []
        xps = sorted(glob.glob(os.path.join(save_dir, "X_chunk_*.npy")))
        yps = sorted(glob.glob(os.path.join(save_dir, "y_chunk_*.npy")))
        assert len(xps)==len(yps) and xps, "Chunk mancanti"
        for xp, yp in zip(xps, yps):
            Xs.append(np.load(xp).astype(np.float32))
            Ys.append(np.load(yp).astype(np.int8).reshape(-1))
        X = np.concatenate(Xs, 0); y = np.concatenate(Ys, 0)
        print(f"Merge → X{X.shape}, y{y.shape}")
        return X, y

 Import da powbal.sequence_utils


## 5) Build chunk 

In [19]:
#  CLEAN CHUNKS DIR (safe) 
import os, glob, shutil

SAVE_DIR = "process_data/event_sequences_chunks"  # stesso percorso che passerai alla funzione

os.makedirs(SAVE_DIR, exist_ok=True)
# opzionale: assert di sicurezza
assert "event_sequences_chunks" in os.path.normpath(SAVE_DIR).split(os.sep), "Percorso SAVE_DIR sospetto"

for pat in ("X_chunk_*.npy", "y_chunk_*.npy", "meta_chunk_*.npz"):
    for p in glob.glob(os.path.join(SAVE_DIR, pat)):
        os.remove(p)
print(f"Cartella pulita: {SAVE_DIR}")


Cartella pulita: process_data/event_sequences_chunks


In [ ]:
clear

In [20]:
def clear_old_chunks(save_dir):
    pats = ["X_chunk_*.npy", "y_chunk_*.npy", "meta_chunk_*.npz"]
    files = []
    for p in pats:
        files.extend(glob.glob(os.path.join(save_dir, p)))
    for f in files:
        try: os.remove(f)
        except FileNotFoundError: pass
    print(f"Removed {len(files)} files from {save_dir}")


In [21]:
# BUILD SEQUENCES — STREAMING VERSION (event-anchored, label-at-center, meta) 

import os, gc, glob
import numpy as np

def build_sequences_chunked_streaming(
    df,
    features,                                # include anche 'y_override' (per estrarre y al centro)
    event_flag_col="scheduled_event",        # = (switch_off_event==1 & posted_to_api==1)
    save_dir="process_data/event_sequences_chunks",
    sequence_length=48,
    label_col="y_override",
    user_col="ca_number",
    datetime_col="parsed_datetime",
    chunk_size=20000,                        # 10k-20k ok; abbassa se hai poca RAM
    save_meta=True,
    extra_meta_cols=("actual_event","reward","appliance","city", "has_event"),  # facoltative ma utili
    verbose=True,
):
    """
    Builds event-centric streaming windows and saves chunks:
      - X_chunk_k.npy: float32 (n_k, T, F_in)  [senza il canale label]
      - y_chunk_k.npy: int8    (n_k,)
      - meta_chunk_k.npz: {fx_col, center_idx, features, user_ids, event_times, ...extra}
    
    NOTE:
    - Builds event-centric streaming windows and saves chunks:to).
    - The label is label_col in the centre of the window; the label channel is removed from X.
    - Categorical columns (e.g. appliance, city) do NOT enter X: they remain in the meta.
    """
    os.makedirs(save_dir, exist_ok=True)
    center = sequence_length // 2

    # basic checks
    if label_col not in features:
        raise ValueError(f"'{label_col}' deve essere incluso in 'features' per poter estrarre y al centro.")

    # index preparation: we separate input features (without labels) and label positions
    idx_label = features.index(label_col)
    feat_in = [c for c in features if c != label_col]          # queste sono le F_in salvate in X
    F_in = len(feat_in)

    # buffer
    X_buf, y_buf, users_buf, times_buf = [], [], [], []
    extras = {c: [] for c in (extra_meta_cols or [])}          # meta extra (facoltativi)
    chunk_id = 0

    # sort by user/time and stream iteratively
    df_sorted = df.sort_values([user_col, datetime_col])
    for uid, g in df_sorted.groupby(user_col, sort=False):
        g = g.reset_index(drop=True)

        # numpy views per velocità/memoria
        data_all = g[features].to_numpy(dtype=np.float32, copy=False)  # include label_col
        data_X   = g[feat_in].to_numpy(dtype=np.float32, copy=False)   # solo canali input
        flags    = g[event_flag_col].to_numpy()
        times    = g[datetime_col].to_numpy()

        # eventuali colonne extra per meta
        extra_arrays = {c: (g[c].to_numpy() if c in g.columns else None) for c in extras.keys()}

        idxs = np.flatnonzero(flags == 1)
        for i in idxs:
            s = i - center
            e = i + center
            if s < 0 or e > len(g):
                continue  # finestra incompleta ai bordi: scarta

            # estraggo finestra input e label al centro
            win_X = data_X[s:e]                            # (T, F_in)  già senza canale label
            y_val = data_all[i, idx_label]                 # label proprio nello slot t=0 (centrale)
            if np.isnan(y_val):
                continue
            y_val = 1 if y_val > 0.5 else 0

            X_buf.append(win_X)
            y_buf.append(y_val)

            if save_meta:
                users_buf.append(uid)
                times_buf.append(times[i])
                for c, arr in extra_arrays.items():
                    if arr is not None:
                        extras[c].append(arr[i])
                    else:
                        extras[c].append(None)

            # flush a chunk
            if len(X_buf) >= chunk_size:
                X_arr = np.asarray(X_buf, dtype=np.float32)
                y_arr = np.asarray(y_buf, dtype=np.int8)
                np.save(os.path.join(save_dir, f"X_chunk_{chunk_id}.npy"), X_arr)
                np.save(os.path.join(save_dir, f"y_chunk_{chunk_id}.npy"), y_arr)

                if save_meta:
                    meta = {
                        "fx_col": event_flag_col,
                        "center_idx": center,
                        "features": np.array(feat_in),            # F_in effettive salvate in X
                        "user_ids": np.array(users_buf),
                        "event_times": np.array(times_buf),
                        "label_col": label_col,
                    }
                    for c, lst in extras.items():
                        meta[c] = np.array(lst, dtype=object)     # object ok per categoriali
                    np.savez(os.path.join(save_dir, f"meta_chunk_{chunk_id}.npz"), **meta)

                if verbose:
                    vals, cnts = np.unique(y_arr, return_counts=True)
                    print(f"chunk {chunk_id:>2} → X{X_arr.shape}, y{y_arr.shape} | dist={dict(zip(vals.tolist(), cnts.tolist()))}")

                # clear buffers (inclusi extras!)
                X_buf.clear(); y_buf.clear(); users_buf.clear(); times_buf.clear()
                for k in extras: extras[k].clear()
                gc.collect()
                chunk_id += 1

    # flush finale
    if X_buf:
        X_arr = np.asarray(X_buf, dtype=np.float32)
        y_arr = np.asarray(y_buf, dtype=np.int8)
        np.save(os.path.join(save_dir, f"X_chunk_{chunk_id}.npy"), X_arr)
        np.save(os.path.join(save_dir, f"y_chunk_{chunk_id}.npy"), y_arr)

        if save_meta:
            meta = {
                "fx_col": event_flag_col,
                "center_idx": center,
                "features": np.array(feat_in),
                "user_ids": np.array(users_buf),
                "event_times": np.array(times_buf),
                "label_col": label_col,
            }
            for c, lst in extras.items():
                meta[c] = np.array(lst, dtype=object)
            np.savez(os.path.join(save_dir, f"meta_chunk_{chunk_id}.npz"), **meta)

        if verbose:
            vals, cnts = np.unique(y_arr, return_counts=True)
            print(f" chunk {chunk_id:>2} → X{X_arr.shape}, y{y_arr.shape} | dist={dict(zip(vals.tolist(), cnts.tolist()))}")

    return



la prima e' quella piu leggera

In [ ]:
# >>> PATCH: build_sequences_chunked_streaming (compat + meta estesi) <<<

import os, gc, numpy as np

def build_sequences_chunked_streaming(
    df,
    features,
    event_flag_col,
    save_dir,
    sequence_length=48,
    label_col="y_override",        # nome della colonna-etichetta dentro 'features'
    override_col=None,             # alias retro-compatibile (se passato, sovrascrive label_col)
    user_col="ca_number",
    datetime_col="parsed_datetime",
    chunk_size=20000,
    save_meta=True,
    verbose=True,
    extras_cols=None               # meta “post-hoc”: verranno salvate se presenti
):
    """
    Crea finestre evento-centriche in streaming e salva tanti chunk piccoli:
      - X_chunk_k.npy : float32 (n_k, T, F)
      - y_chunk_k.npy : int8    (n_k,)
      - meta_chunk_k.npz: {features, center_idx, fx_col, label_col, user_ids, event_times, ...extras}

    Output schema IDENTICO a prima + chiavi meta aggiuntive (backward-compatible).
    """
    os.makedirs(save_dir, exist_ok=True)

    center = int(sequence_length // 2)

    # Consenti alias 'override_col' usato in alcune call precedenti
    if override_col is not None:
        label_col = override_col

    # l'etichetta deve essere tra le features (così la leggiamo al centro)
    if label_col not in features:
        raise ValueError(f"'{label_col}' must be included in 'features' to extract y at the center.")

    idx_label = features.index(label_col)

    # colonne extra da salvare nei meta (solo se esistono nel df)
    if extras_cols is None:
        extras_cols = ["actual_event", "actual_event_flag", "reward", "reward_rate", "appliance", "city"]
    extras_cols = [c for c in extras_cols if c in df.columns]

    # buffer
    X_buf, y_buf, users_buf, times_buf = [], [], [], []
    extras_buf = {c: [] for c in extras_cols}
    chunk_id = 0

    # ordina e itera per utente (streaming)
    df_sorted = df.sort_values([user_col, datetime_col])
    for uid, g in df_sorted.groupby(user_col, sort=False):
        g = g.reset_index(drop=True)

        data  = g[features].to_numpy(dtype=np.float32, copy=False)
        flags = g[event_flag_col].to_numpy()
        times = g[datetime_col].to_numpy()

        idxs = np.flatnonzero(flags == 1)
        for i in idxs:
            s = i - center
            e = i + center
            if s < 0 or e > len(g):
                continue  # finestra incompleta ai bordi

            win   = data[s:e]
            y_val = win[center, idx_label]
            if np.isnan(y_val):
                continue
            y_int = 1 if y_val > 0.5 else 0

            X_buf.append(win)
            y_buf.append(y_int)
            if save_meta:
                users_buf.append(uid)
                times_buf.append(times[i])
                for c in extras_cols:
                    extras_buf[c].append(g.loc[i, c])

            # flush di un chunk
            if len(X_buf) >= chunk_size:
                X_arr = np.asarray(X_buf, dtype=np.float32)
                y_arr = np.asarray(y_buf, dtype=np.int8)

                np.save(os.path.join(save_dir, f"X_chunk_{chunk_id}.npy"), X_arr)
                np.save(os.path.join(save_dir, f"y_chunk_{chunk_id}.npy"), y_arr)

                if save_meta:
                    meta = dict(
                        fx_col=event_flag_col,
                        center_idx=center,
                        features=np.array(features),
                        label_col=label_col,
                        user_ids=np.array(users_buf),
                        event_times=np.array(times_buf),
                    )
                    for c in extras_cols:
                        meta[c] = np.array(extras_buf[c])
                    np.savez(os.path.join(save_dir, f"meta_chunk_{chunk_id}.npz"), **meta)

                if verbose:
                    vals, cnts = np.unique(y_arr, return_counts=True)
                    print(f"chunk {chunk_id} → X{X_arr.shape}, y{y_arr.shape} | dist={dict(zip(vals.tolist(), cnts.tolist()))}")

                chunk_id += 1
                X_buf.clear(); y_buf.clear(); users_buf.clear(); times_buf.clear()
                for c in extras_cols: extras_buf[c].clear()
                gc.collect()

    # flush finale
    if X_buf:
        X_arr = np.asarray(X_buf, dtype=np.float32)
        y_arr = np.asarray(y_buf, dtype=np.int8)

        np.save(os.path.join(save_dir, f"X_chunk_{chunk_id}.npy"), X_arr)
        np.save(os.path.join(save_dir, f"y_chunk_{chunk_id}.npy"), y_arr)

        if save_meta:
            meta = dict(
                fx_col=event_flag_col,
                center_idx=center,
                features=np.array(features),
                label_col=label_col,
                user_ids=np.array(users_buf),
                event_times=np.array(times_buf),
            )
            for c in extras_cols:
                meta[c] = np.array(extras_buf[c])
            np.savez(os.path.join(save_dir, f"meta_chunk_{chunk_id}.npz"), **meta)

        if verbose:
            vals, cnts = np.unique(y_arr, return_counts=True)
            print(f" chunk {chunk_id} → X{X_arr.shape}, y{y_arr.shape} | dist={dict(zip(vals.tolist(), cnts.tolist()))}")

# <<< PATCH END


In [22]:
# cleaning before regenerating chunks
clear_old_chunks(SAVE_DIR)

# CALL
SEQUENCE_LENGTH = 48
SAVE_DIR = "process_data/event_sequences_chunks"

features = [
    "energy_Wh", "reward_rate", "notice_time", "week_in_trial",
    "hour_sin", "hour_cos", "temperature", "wind_u", "wind_v", "precip_mm_log", "ssrd_kwh", "snr_kwh", 
    "y_override",                      # <- label, usata SOLO per leggere y al centro
]

build_sequences_chunked_streaming(
    df=df,
    features=features,
    event_flag_col="scheduled_event",  # = (switch_off_event==1 & posted_to_api==1)
    save_dir=SAVE_DIR,
    sequence_length=SEQUENCE_LENGTH,
    label_col="y_override",
    user_col="ca_number",
    datetime_col="parsed_datetime",
    chunk_size=20000,                  # 10k–20k ok
    save_meta=True,
    extra_meta_cols=("actual_event","reward","appliance","city", "has_event"),
    verbose=True,
)



Removed 0 files from process_data/event_sequences_chunks
chunk  0 → X(20000, 48, 12), y(20000,) | dist={0: 19223, 1: 777}
chunk  1 → X(20000, 48, 12), y(20000,) | dist={0: 18987, 1: 1013}
chunk  2 → X(20000, 48, 12), y(20000,) | dist={0: 19098, 1: 902}
chunk  3 → X(20000, 48, 12), y(20000,) | dist={0: 18921, 1: 1079}
chunk  4 → X(20000, 48, 12), y(20000,) | dist={0: 19370, 1: 630}
chunk  5 → X(20000, 48, 12), y(20000,) | dist={0: 19038, 1: 962}
chunk  6 → X(20000, 48, 12), y(20000,) | dist={0: 19339, 1: 661}
chunk  7 → X(20000, 48, 12), y(20000,) | dist={0: 18976, 1: 1024}
chunk  8 → X(20000, 48, 12), y(20000,) | dist={0: 18359, 1: 1641}
chunk  9 → X(20000, 48, 12), y(20000,) | dist={0: 17744, 1: 2256}
chunk 10 → X(20000, 48, 12), y(20000,) | dist={0: 18234, 1: 1766}
chunk 11 → X(20000, 48, 12), y(20000,) | dist={0: 18834, 1: 1166}
 chunk 12 → X(15893, 48, 12), y(15893,) | dist={0: 15013, 1: 880}


In [23]:

x0 = sorted(glob.glob(os.path.join(SAVE_DIR, "X_chunk_*.npy")))[0]
y0 = x0.replace("X_chunk_", "y_chunk_")
m0 = x0.replace("X_chunk_", "meta_chunk_").replace(".npy", ".npz")

X = np.load(x0); y = np.load(y0); meta = np.load(m0, allow_pickle=True)
print("X", X.shape, "y", y.shape)
print("meta keys:", list(meta.files))
print("label_col:", str(meta.get("label_col", "y_override")))


X (20000, 48, 12) y (20000,)
meta keys: ['fx_col', 'center_idx', 'features', 'user_ids', 'event_times', 'label_col', 'actual_event', 'reward', 'appliance', 'city', 'has_event']
label_col: y_override


In [27]:
# # QC-B: ispeziona il PRIMO chunk salvato
# import os, glob, numpy as np, pandas as pd

# xp = sorted(glob.glob(os.path.join(SAVE_DIR, "X_chunk_*.npy")))[0]
# yp = sorted(glob.glob(os.path.join(SAVE_DIR, "y_chunk_*.npy")))[0]
# mp = sorted(glob.glob(os.path.join(SAVE_DIR, "meta_chunk_*.npz")))[0]

# X = np.load(xp); y = np.load(yp)
# meta = np.load(mp, allow_pickle=True)
# features = list(meta['features'])
# user_ids = np.array(meta['user_ids']) if 'user_ids' in meta else None
# event_times = pd.to_datetime(meta['event_times']) if 'event_times' in meta else None

# center = X.shape[1]//2
# idx_override = features.index('y_override')

# print("X shape:", X.shape, "| y shape:", y.shape)
# print("y unique:", np.unique(y))
# print("NaN in X? ", np.isnan(X).any(), "| NaN in y? ", np.isnan(y).any())
# print("y == override@center ? ", bool(np.all(y == X[:, center, idx_override])))

# # (opzionale) verifica che la sequenza corrisponda a 25 step dell'UTENTE corretto nel df originale
# if user_ids is not None and event_times is not None:
#     step = pd.Timedelta(minutes=30)
#     ok = True
#     for j in np.random.choice(range(len(y)), size=min(10, len(y)), replace=False):
#         uid = user_ids[j]; t0 = event_times[j]
#         mask = (df['ca_number']==uid) & (df['parsed_datetime'].between(t0 - center*step, t0 + (X.shape[1]-center-1)*step))
#         cnt = int(mask.sum())
#         if cnt != X.shape[1]:
#             ok = False; print(f"Window mismatch seq {j}: attesi {X.shape[1]}, trovati {cnt}")
#     print("Window check (10 campioni):", "OK" if ok else "ATTENZIONE")


ValueError: 'y_override' is not in list

In [27]:
# file cleaning
for pat in ["X_chunk_*.npy", "y_chunk_*.npy", "meta_chunk_*.npz"]:
    for p in glob.glob(os.path.join(SAVE_DIR, pat)):
        os.remove(p)


## 6) Global balance & per-chunk

In [24]:
# %% 6) Balance
analyze_class_balance_from_chunks(SAVE_DIR)
analyze_class_balance_per_chunk(SAVE_DIR)

 Bilanciamento globale: N=255893 | Pos=14757 (5.77%)
Chunk  9 | fx=scheduled_event  | N=20000  | Pos=2256   | 11.28%
Chunk 10 | fx=scheduled_event  | N=20000  | Pos=1766   | 8.83%
Chunk  8 | fx=scheduled_event  | N=20000  | Pos=1641   | 8.21%
Chunk 11 | fx=scheduled_event  | N=20000  | Pos=1166   | 5.83%
Chunk 12 | fx=scheduled_event  | N=15893  | Pos=880    | 5.54%
Chunk  3 | fx=scheduled_event  | N=20000  | Pos=1079   | 5.39%
Chunk  7 | fx=scheduled_event  | N=20000  | Pos=1024   | 5.12%
Chunk  1 | fx=scheduled_event  | N=20000  | Pos=1013   | 5.07%
Chunk  5 | fx=scheduled_event  | N=20000  | Pos=962    | 4.81%
Chunk  2 | fx=scheduled_event  | N=20000  | Pos=902    | 4.51%
Chunk  0 | fx=scheduled_event  | N=20000  | Pos=777    | 3.89%
Chunk  6 | fx=scheduled_event  | N=20000  | Pos=661    | 3.31%
Chunk  4 | fx=scheduled_event  | N=20000  | Pos=630    | 3.15%


[{'chunk_id': 9,
  'fx_col': 'scheduled_event',
  'n': 20000,
  'pos': 2256,
  'pos_rate': 0.1128},
 {'chunk_id': 10,
  'fx_col': 'scheduled_event',
  'n': 20000,
  'pos': 1766,
  'pos_rate': 0.0883},
 {'chunk_id': 8,
  'fx_col': 'scheduled_event',
  'n': 20000,
  'pos': 1641,
  'pos_rate': 0.08205},
 {'chunk_id': 11,
  'fx_col': 'scheduled_event',
  'n': 20000,
  'pos': 1166,
  'pos_rate': 0.0583},
 {'chunk_id': 12,
  'fx_col': 'scheduled_event',
  'n': 15893,
  'pos': 880,
  'pos_rate': 0.055370288806392755},
 {'chunk_id': 3,
  'fx_col': 'scheduled_event',
  'n': 20000,
  'pos': 1079,
  'pos_rate': 0.05395},
 {'chunk_id': 7,
  'fx_col': 'scheduled_event',
  'n': 20000,
  'pos': 1024,
  'pos_rate': 0.0512},
 {'chunk_id': 1,
  'fx_col': 'scheduled_event',
  'n': 20000,
  'pos': 1013,
  'pos_rate': 0.05065},
 {'chunk_id': 5,
  'fx_col': 'scheduled_event',
  'n': 20000,
  'pos': 962,
  'pos_rate': 0.0481},
 {'chunk_id': 2,
  'fx_col': 'scheduled_event',
  'n': 20000,
  'pos': 902,
  'pos

In [24]:
_ = analyze_class_balance_per_chunk(SAVE_DIR)


Chunk  9 | fx=scheduled_event  | N=20000  | Pos=2256   | 11.28%
Chunk 10 | fx=scheduled_event  | N=20000  | Pos=1766   | 8.83%
Chunk  8 | fx=scheduled_event  | N=20000  | Pos=1641   | 8.21%
Chunk 11 | fx=scheduled_event  | N=20000  | Pos=1166   | 5.83%
Chunk 12 | fx=scheduled_event  | N=15893  | Pos=880    | 5.54%
Chunk  3 | fx=scheduled_event  | N=20000  | Pos=1079   | 5.39%
Chunk  7 | fx=scheduled_event  | N=20000  | Pos=1024   | 5.12%
Chunk  1 | fx=scheduled_event  | N=20000  | Pos=1013   | 5.07%
Chunk  5 | fx=scheduled_event  | N=20000  | Pos=962    | 4.81%
Chunk  2 | fx=scheduled_event  | N=20000  | Pos=902    | 4.51%
Chunk  0 | fx=scheduled_event  | N=20000  | Pos=777    | 3.89%
Chunk  6 | fx=scheduled_event  | N=20000  | Pos=661    | 3.31%
Chunk  4 | fx=scheduled_event  | N=20000  | Pos=630    | 3.15%


In [25]:
N, P, R = analyze_class_balance_from_chunks(SAVE_DIR)
print(f"Globale: N={N}, Pos={P} ({R:.2%})")


Bilanciamento globale: N=255893 | Pos=14757 (5.77%)
Globale: N=255893, Pos=14757 (5.77%)


## 7) Merge finale per il training (Notebook 02/04)

In [26]:
# %% 7) Merge → X_final / y_final
X_final, y_final = load_all_chunks(SAVE_DIR)
np.save(os.path.join(SAVE_DIR, "X_final.npy"), X_final.astype(np.float32))
np.save(os.path.join(SAVE_DIR, "y_final.npy"), y_final.astype(np.int8))
print("Salvati:", os.path.join(SAVE_DIR, "X_final.npy"), os.path.join(SAVE_DIR, "y_final.npy"))

Merge → X(255893, 48, 12), y(255893,)
Salvati: process_data/event_sequences_chunks/X_final.npy process_data/event_sequences_chunks/y_final.npy


In [48]:
# # %% 8) QC (primo chunk)
# xps = sorted(glob.glob(os.path.join(SAVE_DIR, "X_chunk_*.npy")))
# yps = sorted(glob.glob(os.path.join(SAVE_DIR, "y_chunk_*.npy")))
# if xps:
#     X0 = np.load(xps[0]); y0 = np.load(yps[0])
#     meta0 = np.load(os.path.join(SAVE_DIR, f"meta_chunk_{int(os.path.basename(xps[0]).split('_')[-1].split('.')[0])}.npz"), allow_pickle=True)
#     feats = list(meta0["features"])
#     cidx = int(meta0["center_idx"]) if "center_idx" in meta0 else (X0.shape[1]//2)
#     idx_over = feats.index("override")
#     y_check = (X0[:, cidx, idx_over] > 0.5).astype(np.int8)
#     assert np.array_equal(y0, y_check), "Mismatch: y_chunk diverso da override al centro"
#     print(" QC ok: y al centro coincide con override.")
# else:
#     print("Niente chunk trovati per QC.")

ValueError: 'override' is not in list

In [ ]:
#loader meta

In [ ]:
import numpy as np, glob, os
mp = sorted(glob.glob(os.path.join(SAVE_DIR, "meta_chunk_*.npz")))[0]
meta = np.load(mp, allow_pickle=True)
features_input = list(meta["features"])


### Next steps
Go to **Notebook 02** for: group split by user, random crop (no future) and 3D scaling; then to **Notebook 03** for LSTM.